# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook hoàn chỉnh cho Kaggle. Chạy tuần tự từng bước; các công tắc `RUN_*` ngăn một lần **Run All**
vô tình tiêu hết GPU hoặc mở tập test trước khi chốt cấu hình.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Kaggle: bật Internet trong Settings trước khi chạy.
!pip -q install timm openpyxl thop

import json, os, platform, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

STUDENT_REL = Path("submission/2A202602920_NguyenMinhDuong")
search_roots = [Path.cwd().resolve(), Path("/kaggle/working")]
candidates = []
for base in search_roots:
    if not base.exists():
        continue
    candidates.extend([base, *[p for p in base.iterdir() if p.is_dir()]])
REPO_DIR = next(
    (p for p in candidates if (p / "eval.py").is_file() and (p / STUDENT_REL / "code").is_dir()),
    None,
)
if REPO_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy repo. Hãy clone repo vào /kaggle/working rồi chạy lại cell này."
    )
REPO_DIR = REPO_DIR.resolve()
SUBMISSION_DIR = REPO_DIR / STUDENT_REL
CODE_DIR = SUBMISSION_DIR / "code"
DATA_DIR = (Path("/kaggle/working/deepweeds_data")
            if Path("/kaggle/working").exists() else REPO_DIR / "data")

for path in (str(CODE_DIR), str(REPO_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)
for folder in ("runs", "curves", "predictions", "eval_out"):
    (SUBMISSION_DIR / folder).mkdir(parents=True, exist_ok=True)

print("repo:", REPO_DIR)
print("data:", DATA_DIR)
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu
Ảnh được tải từ Zenodo (~490 MB), kiểm tra MD5 rồi giải nén vào vùng làm việc của Kaggle.
Nhãn fold 0 được tải nguyên bản từ GitHub của tác giả. Cell tự tìm thư mục chứa đủ ảnh, không cần sửa path thủ công.

In [ ]:
import hashlib, urllib.request, zipfile

DATA_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR = DATA_DIR / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
zip_path = DATA_DIR / "images.zip"
image_url = "https://zenodo.org/records/7939060/files/images.zip?download=1"
if not zip_path.exists():
    print("Đang tải images.zip...")
    urllib.request.urlretrieve(image_url, zip_path)

h = hashlib.md5()
with zip_path.open("rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

extract_marker = DATA_DIR / ".images_extracted"
if not extract_marker.exists():
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATA_DIR)
    extract_marker.touch()

base_url = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ("labels", "train_subset0", "val_subset0", "test_subset0"):
    target = LABELS_DIR / f"{name}.csv"
    if not target.exists():
        urllib.request.urlretrieve(f"{base_url}/{name}.csv", target)

jpgs = [*DATA_DIR.rglob("*.jpg"), *DATA_DIR.rglob("*.JPG")]
parent_counts = pd.Series([str(path.parent) for path in jpgs]).value_counts()
if parent_counts.empty:
    raise FileNotFoundError(f"Không tìm thấy ảnh JPG sau khi giải nén ở {DATA_DIR}")
IMAGES_DIR = Path(parent_counts.index[0])
print("images:", IMAGES_DIR, "| jpg:", parent_counts.iloc[0])
print("labels:", LABELS_DIR)

In [ ]:
# Toàn bộ output nhỏ nằm trong submission để có thể git add/commit sau phiên Kaggle.
COMMON = dict(
    images_dir=str(IMAGES_DIR),
    labels_dir=str(LABELS_DIR),
    out_dir=str(SUBMISSION_DIR / "runs"),
    pred_dir=str(SUBMISSION_DIR / "predictions"),
    num_workers=2,
)
print(COMMON)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import dataset

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

distribution = pd.DataFrame({
    split: frame["Label"].value_counts().reindex(range(dataset.NUM_CLASSES), fill_value=0)
    for split, frame in {"train": train_df, "val": val_df, "test": test_df}.items()
})
distribution.index = dataset.CLASS_NAMES
display(distribution)
ax = distribution.plot.bar(figsize=(13, 5), title="DeepWeeds fold 0 — class distribution")
ax.set(xlabel="Class", ylabel="Images")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(dataset.NUM_CLASSES, 3, figsize=(9, 24))
for label, class_name in enumerate(dataset.CLASS_NAMES):
    examples = train_df[train_df["Label"] == label].sample(3, random_state=0)
    for axis, (_, row) in zip(axes[label], examples.iterrows()):
        axis.imshow(Image.open(IMAGES_DIR / row["Filename"]).convert("RGB"))
        axis.set_title(class_name)
        axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra nhanh pipeline. Bật kiểm tra overfit sau khi đã xem ảnh augmentation.
import math
import torch.nn.functional as F
from dataset import IMAGENET_MEAN, IMAGENET_STD
from model import build_model
from train import set_seed

set_seed(0)
transform = dataset.build_transforms(train=True, img_size=224, aug="basic")
check_loader = dataset.make_loader(
    train_df.head(64), IMAGES_DIR, transform, batch_size=8,
    train=False, num_workers=0,
)
images, labels, names = next(iter(check_loader))
model_check = build_model("resnet50", pretrained=False, num_classes=9, init="scratch")
with torch.inference_mode():
    initial_loss = F.cross_entropy(model_check(images), labels).item()
print(f"Initial CE = {initial_loss:.4f}; reference ln(9) = {math.log(9):.4f}")
from losses import FocalLoss
probe_logits = torch.randn(16, 9)
probe_labels = torch.randint(0, 9, (16,))
assert torch.allclose(
    FocalLoss(gamma=0)(probe_logits, probe_labels),
    F.cross_entropy(probe_logits, probe_labels), atol=1e-6,
), "FocalLoss gamma=0 phải trùng CrossEntropyLoss"

mean = torch.tensor(IMAGENET_MEAN)[:, None, None]
std = torch.tensor(IMAGENET_STD)[:, None, None]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label, name in zip(axes.flat, images, labels, names):
    axis.imshow((image.cpu() * std + mean).permute(1, 2, 0).clamp(0, 1))
    axis.set_title(f"{dataset.CLASS_NAMES[int(label)]}\n{name}")
    axis.axis("off")
plt.tight_layout()
plt.show()

RUN_OVERFIT_CHECK = False
if RUN_OVERFIT_CHECK:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tiny_x, tiny_y = images[:8].to(device), labels[:8].to(device)
    tiny_model = build_model("resnet50", pretrained=False, num_classes=9, init="scratch").to(device)
    optimizer = torch.optim.Adam(tiny_model.parameters(), lr=3e-4)
    tiny_model.train()
    for step in range(300):
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(tiny_model(tiny_x), tiny_y)
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(step, float(loss))
    assert float(loss) < 0.1, "Chưa overfit được batch nhỏ; kiểm tra pipeline trước khi train thật."
del model_check
torch.cuda.empty_cache()

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config, run

BACKBONES = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "efficientnet_b0"),
]
RUN_BACKBONE_SWEEP = False  # bật sau khi pipeline checks đạt

backbone_results = []
if RUN_BACKBONE_SWEEP:
    for exp_id, backbone in BACKBONES:
        backbone_results.append(run(Config(
            exp_id=exp_id, backbone=backbone, seed=0, **COMMON
        )))
else:
    for path in (SUBMISSION_DIR / "runs").glob("B*/seed*/summary.json"):
        backbone_results.append(json.loads(path.read_text(encoding="utf-8")))

backbones_df = pd.DataFrame(backbone_results)
if not backbones_df.empty:
    backbones_df = backbones_df.sort_values("val_macro_f1", ascending=False)
display(backbones_df)
SELECTED_BACKBONE = (backbones_df.iloc[0]["backbone"]
                     if not backbones_df.empty else "resnet50")
print("Backbone dùng cho ablation (chỉ chốt theo val):", SELECTED_BACKBONE)

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Bốn trục A/B/C/F và một cấu hình kết hợp. Mỗi T01..T10 chỉ đổi một yếu tố so với T00.
TRAINING_EXPERIMENTS = [
    ("T00", {}),
    ("T01", {"init": "scratch"}),                         # A: initialization
    ("T02", {"init": "frozen"}),                          # A
    ("T03", {"aug": "color"}),                            # B: augmentation
    ("T04", {"aug": "randaug"}),                          # B
    ("T05", {"mix": "cutmix", "mix_alpha": 1.0}),         # B
    ("T06", {"loss": "ls", "label_smoothing": 0.1}),      # C: loss
    ("T07", {"loss": "focal", "focal_gamma": 2.0}),       # C
    ("T08", {"loss": "ce_weighted", "class_weight_beta": 0.0}),  # C
    ("T09", {"sampler": "balanced"}),                     # D: sampling
    ("T10", {"ema_decay": 0.999}),                        # F: regularization
    ("T11", {"mix": "cutmix", "loss": "ls",
             "label_smoothing": 0.1, "ema_decay": 0.999}), # combination
]
RUN_TRAINING_ABLATIONS = False

training_results = []
if RUN_TRAINING_ABLATIONS:
    for exp_id, changes in TRAINING_EXPERIMENTS:
        training_results.append(run(Config(
            exp_id=exp_id, backbone=SELECTED_BACKBONE, seed=0,
            **COMMON, **changes
        )))
else:
    for path in (SUBMISSION_DIR / "runs").glob("T*/seed*/summary.json"):
        training_results.append(json.loads(path.read_text(encoding="utf-8")))

training_df = pd.DataFrame(training_results)
if not training_df.empty:
    baseline = training_df.loc[training_df["exp_id"] == "T00", "val_macro_f1"]
    training_df["delta_vs_T00"] = (
        training_df["val_macro_f1"] - baseline.iloc[0] if len(baseline) else np.nan
    )
    training_df = training_df.sort_values("val_macro_f1", ascending=False)
display(training_df)
BEST_TRAIN_EXP = (training_df.iloc[0]["exp_id"] if not training_df.empty else "T11")
print("Recipe ứng viên (phải chốt theo val):", BEST_TRAIN_EXP)

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# Bật sau khi đã chọn BEST_TRAIN_EXP. Kết quả chỉ dùng validation.
from eval import compute_metrics
from inference import (
    aggregate_views, apply_temperature, fit_temperature, fuse_conv_bn,
    predict_logits, view_hflip,
)
from benchmark import latency_report, tta_latency
from model import build_model

RUN_INFERENCE_STUDY = False
inference_records, latency_records = [], []

def metric_row(exp_id, method, probs, y_true, k):
    m = compute_metrics(y_true, probs.argmax(1), probs)
    return dict(exp_id=exp_id, method=method, K=k, macro_f1_val=m["macro_f1"],
                top1_val=m["top1"], ece_val=m["ece"])

if RUN_INFERENCE_STUDY:
    summary_path = next((SUBMISSION_DIR / "runs" / BEST_TRAIN_EXP).glob("seed0/summary.json"))
    config_path = summary_path.parent / "config.json"
    cfg = Config(**json.loads(config_path.read_text(encoding="utf-8")))
    checkpoint = torch.load(summary_path.parent / "best.pt", map_location="cpu")
    inference_model = build_model(
        cfg.backbone, pretrained=False, num_classes=9,
        drop_rate=cfg.drop_rate, init="scratch",
    )
    inference_model.load_state_dict(checkpoint["model"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inference_model.to(device).eval()
    val_loader = dataset.make_loader(
        val_df, IMAGES_DIR, dataset.build_transforms(False, cfg.img_size, cfg.aug),
        cfg.batch_size, train=False, num_workers=cfg.num_workers,
    )

    names, y_val, logits_1 = predict_logits(inference_model, val_loader, device)
    names_flip, y_flip, logits_flip = predict_logits(
        inference_model, val_loader, device, view=view_hflip
    )
    assert names == names_flip and np.array_equal(y_val, y_flip)
    probs_1 = aggregate_views([logits_1], "prob")
    probs_flip_prob = aggregate_views([logits_1, logits_flip], "prob")
    probs_flip_logit = aggregate_views([logits_1, logits_flip], "logit")
    inference_records.extend([
        metric_row("I00", "1-view", probs_1, y_val, 1),
        metric_row("I01", "hflip, probability mean", probs_flip_prob, y_val, 2),
        metric_row("I02", "hflip, logit mean", probs_flip_logit, y_val, 2),
    ])

    _, y_vflip, logits_vflip = predict_logits(
        inference_model, val_loader, device, view=lambda x: torch.flip(x, dims=(-2,))
    )
    _, y_hvflip, logits_hvflip = predict_logits(
        inference_model, val_loader, device, view=lambda x: torch.flip(x, dims=(-2, -1))
    )
    assert np.array_equal(y_val, y_vflip) and np.array_equal(y_val, y_hvflip)
    inference_records.append(metric_row(
        "I03", "4 flips, probability mean",
        aggregate_views([logits_1, logits_flip, logits_vflip, logits_hvflip], "prob"),
        y_val, 4,
    ))

    scale_logits, supported_sizes = [], []
    for size in (224, 256, 288):
        try:
            _, labels_scale, logits_scale = predict_logits(
                inference_model, val_loader, device,
                view=lambda x, s=size: torch.nn.functional.interpolate(
                    x, size=(s, s), mode="bilinear", align_corners=False, antialias=True
                ),
            )
        except (RuntimeError, AssertionError) as exc:
            print(f"Bỏ qua scale {size} cho {cfg.backbone}: {exc}")
            continue
        assert np.array_equal(y_val, labels_scale)
        scale_logits.append(logits_scale)
        supported_sizes.append(size)
    if len(scale_logits) > 1:
        inference_records.append(metric_row(
            "I04", f"multi-scale {supported_sizes}",
            aggregate_views(scale_logits, "prob"), y_val, len(scale_logits)
        ))

    temperature = fit_temperature(logits_1, y_val)
    inference_records.append(metric_row(
        "I07", f"temperature T={temperature:.4f}",
        apply_temperature(logits_1, temperature), y_val, 1
    ))

    fused_model = fuse_conv_bn(inference_model).to(device)
    _, y_fused, logits_fused = predict_logits(fused_model, val_loader, device)
    inference_records.append(metric_row(
        "I08", "Conv-BN fusion", aggregate_views([logits_fused]), y_fused, 1
    ))

    for batch in (1, 32):
        latency_records.append({
            "configuration": "I00", "fused_bn": False,
            **latency_report(inference_model, batch, cfg.img_size, dtype="amp",
                             device=str(device), warmup=10, iters=100),
        })
    latency_records.append({
        "configuration": "I01", "fused_bn": False,
        **tta_latency(inference_model, 2, batch_size=1, img_size=cfg.img_size,
                      dtype="amp", device=str(device), warmup=10, iters=100),
    })
    latency_records.append({
        "configuration": "I08", "fused_bn": True,
        **latency_report(fused_model, 1, cfg.img_size, dtype="amp",
                         device=str(device), warmup=10, iters=100),
    })
    pd.DataFrame(inference_records).to_csv(
        SUBMISSION_DIR / "runs/inference_results.csv", index=False
    )
    pd.DataFrame(latency_records).to_csv(
        SUBMISSION_DIR / "runs/latency_results.csv", index=False
    )

inference_df = (
    pd.DataFrame(inference_records) if inference_records
    else pd.read_csv(SUBMISSION_DIR / "runs/inference_results.csv")
    if (SUBMISSION_DIR / "runs/inference_results.csv").exists() else pd.DataFrame()
)
latency_df = (
    pd.DataFrame(latency_records) if latency_records
    else pd.read_csv(SUBMISSION_DIR / "runs/latency_results.csv")
    if (SUBMISSION_DIR / "runs/latency_results.csv").exists() else pd.DataFrame()
)
display(inference_df)
display(latency_df)

## Bước 4 — Chung kết: ≥ 3 seed, test đúng một lần mỗi seed
1. Chốt backbone, recipe và inference hoàn toàn bằng **val**.
2. Gõ chuỗi xác nhận trong cell dưới rồi chạy cấu hình cuối và mốc với cùng ba seed.
3. Không chạy lại cell sau khi đã xem test. Nếu phát hiện lỗi, ghi lại trong báo cáo.

In [ ]:
# Chỉ cần đổi hai công tắc này sau khi đã chốt validation; recipe được lấy tự động.
RUN_FINAL_TEST = False
FINAL_TEST_CONFIRMATION = ""  # điền chính xác: LOCKED_FROM_VALIDATION

best_config_files = sorted(
    (SUBMISSION_DIR / "runs" / BEST_TRAIN_EXP).glob("seed*/config.json")
)
best_val_config = (
    json.loads(best_config_files[0].read_text(encoding="utf-8"))
    if best_config_files else None
)
recipe_fields = (
    "backbone", "init", "drop_rate", "img_size", "aug", "sampler", "mix", "mix_alpha",
    "loss", "label_smoothing", "focal_gamma", "class_weight_beta", "epochs", "batch_size",
    "lr_backbone", "lr_head", "weight_decay", "warmup_epochs", "ema_decay", "amp",
)
FINAL_RECIPE = (
    {key: best_val_config[key] for key in recipe_fields}
    if best_val_config is not None else None
)

# Temperature scaling chỉ được bật tự động nếu ECE validation giảm so với I00.
FINAL_USE_TEMPERATURE = False
if not inference_df.empty and {"I00", "I07"} <= set(inference_df["exp_id"]):
    ece_by_id = inference_df.set_index("exp_id")["ece_val"]
    FINAL_USE_TEMPERATURE = bool(ece_by_id["I07"] < ece_by_id["I00"])

print("Recipe khóa từ validation:", BEST_TRAIN_EXP, FINAL_RECIPE)
if FINAL_RECIPE is None:
    print("Chưa có recipe; chạy Bước 2 trước khi mở khóa test.")
print("Temperature scaling:", FINAL_USE_TEMPERATURE)

if RUN_FINAL_TEST:
    assert FINAL_TEST_CONFIRMATION == "LOCKED_FROM_VALIDATION"
    assert FINAL_RECIPE is not None, "Chưa có recipe thắng từ validation"
    from eval import save_predictions
    from inference import fit_temperature, apply_temperature

    for seed in (0, 1, 2):
        final_cfg = Config(
            exp_id="F01", seed=seed, save_test_predictions=True,
            **COMMON, **FINAL_RECIPE,
        )
        run(final_cfg)

        baseline_cfg = Config(
            exp_id="T00", backbone=FINAL_RECIPE["backbone"], seed=seed,
            save_test_predictions=True, **COMMON,
        )
        run(baseline_cfg)

        if FINAL_USE_TEMPERATURE:
            final_run_dir = Path(final_cfg.out_dir) / "F01" / f"seed{seed}"
            val_logits = np.load(final_run_dir / "val_logits.npy")
            test_logits = np.load(final_run_dir / "test_logits.npy")
            val_uncal = pd.read_csv(Path(final_cfg.pred_dir) / f"F01_seed{seed}_val.csv")
            test_uncal = pd.read_csv(Path(final_cfg.pred_dir) / f"F01_seed{seed}_test.csv")
            shutil.copy2(
                Path(final_cfg.pred_dir) / f"F01_seed{seed}_test.csv",
                Path(final_cfg.pred_dir) / f"F01uncal_seed{seed}_test.csv",
            )
            temperature = fit_temperature(val_logits, val_uncal["y_true"].to_numpy())
            save_predictions(
                Path(final_cfg.pred_dir) / f"F01_seed{seed}_val.csv",
                val_uncal["Filename"], val_uncal["y_true"],
                apply_temperature(val_logits, temperature),
            )
            save_predictions(
                Path(final_cfg.pred_dir) / f"F01_seed{seed}_test.csv",
                test_uncal["Filename"], test_uncal["y_true"],
                apply_temperature(test_logits, temperature),
            )
            print(f"seed={seed}: fitted T={temperature:.5f} on val")
else:
    print("Test đang khóa. Chỉ mở sau khi đã chốt mọi lựa chọn bằng validation.")

In [ ]:
# Tính chỉ số đúng định nghĩa của lớp khi đủ prediction files.
pred_dir = SUBMISSION_DIR / "predictions"
score_jobs = [
    ("F01", str(pred_dir / "F01_seed*_test.csv")),
    ("T00", str(pred_dir / "T00_seed*_test.csv")),
]
for tag, pattern in score_jobs:
    if list(pred_dir.glob(f"{tag}_seed*_test.csv")):
        subprocess.run([
            sys.executable, str(REPO_DIR / "eval.py"), "score",
            "--pred", pattern,
            "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
            "--labels", str(LABELS_DIR / "labels.csv"),
            "--tag", tag,
            "--out", str(SUBMISSION_DIR / "eval_out"),
        ], check=True)
    else:
        print("Bỏ qua score vì chưa có:", pattern)

In [ ]:
# Tự chấm phần I sau lần test cuối.
if (list(pred_dir.glob("F01_seed*_test.csv"))
        and list(pred_dir.glob("T00_seed*_test.csv"))):
    command = [
        sys.executable, str(REPO_DIR / "eval.py"), "grade",
        "--final", str(pred_dir / "F01_seed*_test.csv"),
        "--baseline", str(pred_dir / "T00_seed*_test.csv"),
        "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
        "--labels", str(LABELS_DIR / "labels.csv"),
        "--out", str(SUBMISSION_DIR / "eval_out"),
    ]
    if list(pred_dir.glob("F01uncal_seed*_test.csv")):
        command.extend(["--uncal", str(pred_dir / "F01uncal_seed*_test.csv")])
    if list(pred_dir.glob("F01_seed*_val.csv")):
        command.extend([
            "--final-val", str(pred_dir / "F01_seed*_val.csv"),
            "--val-csv", str(LABELS_DIR / "val_subset0.csv"),
        ])
    subprocess.run(command, check=True)
else:
    print("Chưa có đủ F01 và T00 test predictions; chưa chạy grade.")

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# Tổng hợp duy nhất từ artifact đã chạy thật; không tạo số giả.
from eval import read_pred, compute_metrics

run_rows = []
for summary_path in (SUBMISSION_DIR / "runs").glob("*/seed*/summary.json"):
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    config_path = summary_path.parent / "config.json"
    config = json.loads(config_path.read_text(encoding="utf-8"))
    run_rows.append({**summary, **{f"cfg_{key}": value for key, value in config.items()}})
all_runs = pd.DataFrame(run_rows)

backbone_columns = [
    "exp_id", "backbone", "weight_tag", "params_m", "gmacs", "resolution",
    "epochs", "seed", "macro_f1_val", "top1_val", "seconds_per_epoch",
    "latency_batch1_ms", "notes",
]
backbone_rows = []
if not all_runs.empty:
    for _, row in all_runs[all_runs["exp_id"].str.startswith("B")].iterrows():
        backbone_rows.append({
            "exp_id": row["exp_id"], "backbone": row["backbone"],
            "weight_tag": row["weight_tag"], "params_m": row["params_m"],
            "gmacs": row["gmacs"], "resolution": row["cfg_img_size"],
            "epochs": row["cfg_epochs"], "seed": row["seed"],
            "macro_f1_val": row["val_macro_f1"], "top1_val": row["val_top1"],
            "seconds_per_epoch": row["seconds_per_epoch"],
            "latency_batch1_ms": np.nan, "notes": "",
        })
backbone_sheet = pd.DataFrame(backbone_rows, columns=backbone_columns)

axis_by_exp = {
    "T00": "Baseline", "T01": "A-init", "T02": "A-init",
    "T03": "B-augmentation", "T04": "B-augmentation", "T05": "B-Mix/CutMix",
    "T06": "C-loss", "T07": "C-loss", "T08": "C-loss",
    "T09": "D-sampler", "T10": "F-EMA", "T11": "Combination",
}
training_columns = [
    "exp_id", "backbone", "changed_axis", "difference_from_T00", "seed",
    "macro_f1_val", "top1_val", "delta_vs_T00", "f1_chinee_apple",
    "f1_snake_weed", "notes",
]
training_rows = []
if not all_runs.empty:
    train_runs = all_runs[all_runs["exp_id"].str.startswith("T")]
    baseline = train_runs[train_runs["exp_id"] == "T00"]
    baseline_f1 = baseline.iloc[0]["val_macro_f1"] if len(baseline) else np.nan
    compare_keys = [
        "cfg_init", "cfg_aug", "cfg_sampler", "cfg_mix", "cfg_loss",
        "cfg_label_smoothing", "cfg_focal_gamma", "cfg_class_weight_beta",
        "cfg_lr_backbone", "cfg_lr_head", "cfg_weight_decay", "cfg_ema_decay",
        "cfg_img_size", "cfg_epochs",
    ]
    baseline_cfg = baseline.iloc[0] if len(baseline) else None
    for _, row in train_runs.iterrows():
        differences = []
        if baseline_cfg is not None:
            for key in compare_keys:
                if row.get(key) != baseline_cfg.get(key):
                    differences.append(f"{key.removeprefix('cfg_')}={row.get(key)}")
        val_pred_path = SUBMISSION_DIR / "predictions" / f"{row['exp_id']}_seed{int(row['seed'])}_val.csv"
        f1_chinee = f1_snake = np.nan
        if val_pred_path.exists():
            prediction = read_pred(str(val_pred_path))
            metrics = compute_metrics(prediction.y_true, prediction.y_pred, prediction.probs)
            f1_chinee, f1_snake = metrics["f1"][0], metrics["f1"][7]
        training_rows.append({
            "exp_id": row["exp_id"], "backbone": row["backbone"],
            "changed_axis": axis_by_exp.get(row["exp_id"], "Custom"),
            "difference_from_T00": "; ".join(differences) or "baseline",
            "seed": row["seed"], "macro_f1_val": row["val_macro_f1"],
            "top1_val": row["val_top1"],
            "delta_vs_T00": row["val_macro_f1"] - baseline_f1,
            "f1_chinee_apple": f1_chinee, "f1_snake_weed": f1_snake,
            "notes": "",
        })
training_sheet = pd.DataFrame(training_rows, columns=training_columns)

inference_columns = [
    "exp_id", "method", "model_checkpoint", "K", "macro_f1_val", "top1_val",
    "ece_val", "p50_ms_batch1", "p95_ms_batch1", "p99_ms_batch1",
    "images_per_s", "relative_cost_vs_I00",
]
inference_sheet = pd.DataFrame(columns=inference_columns)
if not inference_df.empty:
    inference_sheet = inference_df.copy()
    inference_sheet["model_checkpoint"] = BEST_TRAIN_EXP
    batch1_latency = latency_df[latency_df["batch"] == 1] if not latency_df.empty else pd.DataFrame()
    if not batch1_latency.empty:
        latency_map = batch1_latency.drop_duplicates("configuration").set_index("configuration")
        for stat in ("p50", "p95", "p99", "images_per_s"):
            inference_sheet[f"{stat}_ms_batch1" if stat != "images_per_s" else stat] = (
                inference_sheet["exp_id"].map(latency_map[stat])
            )
        base_p50 = latency_map.loc["I00", "p50"] if "I00" in latency_map.index else np.nan
        inference_sheet["relative_cost_vs_I00"] = inference_sheet["p50_ms_batch1"] / base_p50
    inference_sheet = inference_sheet.reindex(columns=inference_columns)

final_rows, per_class_rows = [], []
summary_lookup = {}
if not all_runs.empty:
    summary_lookup = {
        (str(row["exp_id"]), int(row["seed"])): row
        for _, row in all_runs.iterrows()
    }
for path in sorted((SUBMISSION_DIR / "predictions").glob("*_seed*_test.csv")):
    if "uncal" in path.name:
        continue
    pred = read_pred(str(path))
    metrics = compute_metrics(pred.y_true, pred.y_pred, pred.probs)
    exp_id = path.name.split("_seed")[0]
    run_summary = summary_lookup.get((exp_id, int(pred.seed)))
    final_rows.append({
        "exp_id": exp_id,
        "configuration": (run_summary["backbone"] if run_summary is not None else ""),
        "seed": pred.seed,
        "macro_f1_val": (run_summary["val_macro_f1"] if run_summary is not None else np.nan),
        "macro_f1_test": metrics["macro_f1"],
        "top1_test": metrics["top1"],
        "ece_test": metrics["ece"],
    })
    for label, class_name in enumerate(dataset.CLASS_NAMES):
        per_class_rows.append({
            "exp_id": exp_id, "seed": pred.seed, "class": class_name,
            "support": int(metrics["support"][label]),
            "precision": metrics["precision"][label],
            "recall": metrics["recall"][label],
            "f1": metrics["f1"][label],
        })
final_sheet = pd.DataFrame(final_rows, columns=[
    "exp_id", "configuration", "seed", "macro_f1_val",
    "macro_f1_test", "top1_test", "ece_test",
])
if not final_sheet.empty:
    aggregate = final_sheet.groupby("exp_id").agg(
        macro_f1_test_mean=("macro_f1_test", "mean"),
        macro_f1_test_std=("macro_f1_test", lambda x: x.std(ddof=1)),
        top1_test_mean=("top1_test", "mean"),
        top1_test_std=("top1_test", lambda x: x.std(ddof=1)),
        ece_test_mean=("ece_test", "mean"),
        ece_test_std=("ece_test", lambda x: x.std(ddof=1)),
    ).reset_index()
    final_sheet = final_sheet.merge(aggregate, on="exp_id", how="left")
per_class_sheet = pd.DataFrame(per_class_rows, columns=[
    "exp_id", "seed", "class", "support", "precision", "recall", "f1",
])

latency_columns = [
    "configuration", "gpu", "dtype", "batch", "img_size", "fused_bn",
    "p50", "p95", "p99", "images_per_s", "torch",
]
latency_sheet = (
    latency_df.reindex(columns=latency_columns)
    if not latency_df.empty else pd.DataFrame(columns=latency_columns)
)
summary_sheet = (
    all_runs.sort_values("val_macro_f1", ascending=False).head(10)
    if not all_runs.empty else pd.DataFrame(columns=["exp_id", "val_macro_f1"])
)

sheets = {
    "Backbones": backbone_sheet,
    "Training": training_sheet,
    "Inference": inference_sheet,
    "Final": final_sheet,
    "PerClass": per_class_sheet,
    "Latency": latency_sheet,
    "Summary": summary_sheet,
}
xlsx_path = SUBMISSION_DIR / "results.xlsx"
if all_runs.empty:
    print("Chưa có lượt train thật; chưa tạo results.xlsx.")
else:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name, index=False)
            ws = writer.book[name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for column in ws.columns:
                width = min(45, max(12, max(len(str(cell.value or "")) for cell in column) + 2))
                ws.column_dimensions[column[0].column_letter].width = width

if not all_runs.empty:
    missing_curves = [
        row["curve"] for _, row in all_runs.iterrows()
        if not (SUBMISSION_DIR / row["curve"]).is_file()
    ]
    if missing_curves:
        raise FileNotFoundError(f"Thiếu curve files: {missing_curves}")

if not all_runs.empty:
    print("Đã ghi:", xlsx_path)